# 蜃景 × Colab 部署（空白可填版）

完整一条龙：**ComfyUI + 后端 + 公网地址 + 抗断线持久化**——但 **出图 / 出片模型默认全空、不预设、不下载**。
你在下面 **§1 配置** cell 自己填要用的底模 / 出片 workflow / 要下载的权重清单。

**用法**：① 运行时选 **GPU** ② 在 §1 填好你的出图/出片 ③ 菜单「代码执行程序 → 全部运行」。
断线/被回收后再点一次 **Run all** 即可（模型、依赖持久化在 Drive，不重下）。

**Secrets（右侧 🔑，按需）**：用 CivitAI 底模加 `CIVITAI_TOKEN`；用 gated 的 HF 模型加 `HF_TOKEN`。


In [ ]:
# ===== §1 配置：出图 / 出片 你自己填（默认全空：什么都不预设、不下载）=====
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'
BRANCH   = 'main'
DEEPSEEK_KEY = ''            # 出图/出片不需要 LLM，可留空

# ---- 出图底模（t2i）。不填 = 不配出图。二选一：----
IMG_KIND = ''                # 'checkpoint'(CivitAI 全合一) / 'unet'(HF UNET-only) / '' (不配出图)
IMG_FILE = ''                # 落地存名，如 'my-model.safetensors'（COMFYUI_FLUX_CKPT/UNET 自动跟随）
#  (A) CivitAI 全合一 checkpoint：填版本号；token 走 Secret CIVITAI_TOKEN（不写进代码）
IMG_CIVITAI_VERSION = ''     # 下载链接 models/ 后的数字
IMG_CIVITAI_FILEID  = ''     # 该版本多文件变体时填，否则留空
#  (B) HF UNET-only：填仓库（IMG_FILE 为其中的 .safetensors 名）
IMG_HF_REPO = ''             # 如 'black-forest-labs/FLUX.1-dev'
VID_T2I_WORKFLOW = ''        # 出图模板路径；空=checkpoint 用自带 t2i_checkpoint_template.json / unet 用 flux 自带

# ---- 出片（i2v）workflow。不填 = 不配出片。----
VID_WORKFLOW = ''            # 你的 i2v 模板路径，如 'comfyui_workflows/i2v_fp8_template.json'；空=不配

# ---- 要下载的模型清单（你自己加；默认空 = 什么都不下）----
#   每条 = (HF仓库, 仓库内路径, 落到 ComfyUI/models 的子目录)
#   出片(fp8)例：
#     ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors','unet')
#     ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors','unet')
#     ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','clip')
#     ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/vae/wan_2.1_vae.safetensors','vae')
MODELS_TO_DOWNLOAD = [
    # 在这里加你要的权重行
]

print('配置已读 → 出图: kind=%r file=%r | 出片 workflow=%r | 待下模型 %d 个'
      % (IMG_KIND, IMG_FILE, VID_WORKFLOW, len(MODELS_TO_DOWNLOAD)))


In [ ]:
# ===== §2 挂 Drive + 持久化路径 + GPU 探测（只报告，不强制任何模型）=====
import os, torch as _t
try:
    from google.colab import userdata
    os.environ['CIVITAI_TOKEN'] = userdata.get('CIVITAI_TOKEN') or ''
except Exception: pass
try:
    from google.colab import userdata as _ud
    os.environ['HF_TOKEN'] = _ud.get('HF_TOKEN') or ''
except Exception: pass

# GPU 探测：只给出"建议精度/启动参数"，不替你选模型
if _t.cuda.is_available():
    cc = _t.cuda.get_device_capability(0); name = _t.cuda.get_device_name(0)
    vram = _t.cuda.get_device_properties(0).total_memory/(1024**3)
    native_fp8 = (cc[0] > 8) or (cc[0] == 8 and cc[1] >= 9)
    hv = '1' if (native_fp8 and vram >= 70) else '0'   # 原生FP8+80G 才开 highvram
    os.environ['MIRAGE_HIGHVRAM'] = hv
    os.environ['MIRAGE_USE_SAGE'] = '0' if native_fp8 else '1'  # 原生FP8(Blackwell/Hopper)上 sage 对 Wan 有噪声 → 关
    print(f'🖥 GPU: {name} | sm{cc[0]}.{cc[1]} | {vram:.0f}G | 原生FP8={native_fp8}')
    print(f'   建议: 原生FP8 卡→fp8 模板 / A100-80G→fp16 / <24G→GGUF（你在 §1 自己选 workflow 与权重）')
    print(f'   ComfyUI 启动参数: highvram={hv} use_sage={os.environ["MIRAGE_USE_SAGE"]}')
else:
    os.environ['MIRAGE_HIGHVRAM']='0'; os.environ['MIRAGE_USE_SAGE']='1'
    print('⚠️ 没检测到 GPU！代码执行程序 → 更改运行时类型 → 选 GPU')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
CACHE=DRIVE+'/mirage_models'; TOOLS=DRIVE+'/mirage_tools'; PIP_CACHE=TOOLS+'/pip_cache'
for p in (CACHE, TOOLS, PIP_CACHE): os.makedirs(p, exist_ok=True)
os.environ['PIP_CACHE_DIR']=PIP_CACHE
os.environ['HF_HOME']=TOOLS+'/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('环境就绪 | 模型持久化目录:', CACHE)


In [ ]:
# ===== §3 拉取/更新仓库 =====
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
print('仓库就绪 @', BRANCH)


In [ ]:
# ===== §4 装 ComfyUI + 自定义节点（不下载任何模型）=====
!bash /content/mirage/colab/setup.sh


In [ ]:
# ===== §5 软链 Drive（不删数据）+ 按你 §1 的清单下模型（空=什么都不下）=====
import os, sys, subprocess
sys.path.insert(0, '/content/mirage/colab'); import persist
persist.link_models(CACHE, '/content/ComfyUI/models',
    ['unet','checkpoints','clip','vae','audio_encoders','loras','pulid','text_encoders','diffusion_models','insightface','upscale_models'])

# 通用下载器：mirror colab/download_models.sh 的 get()——find -L 穿透 Drive 软链做扁平化
_GET = r"""get(){ local M="/content/ComfyUI/models/$3"; mkdir -p "$M"; local b=$(basename "$2");
  if [ -s "$M/$b" ]; then echo "[skip] $b"; return; fi; echo "[get ] $b";
  hf download "$1" "$2" --local-dir "$M" >/dev/null 2>&1 || true;
  if [ ! -s "$M/$b" ]; then local g=$(find -L "$M" -type f -name "$b" 2>/dev/null|head -1); [ -n "$g" ] && mv -f "$g" "$M/$b"; fi; }"""
def hf_get(repo, path, sub):
    subprocess.run(['bash','-lc', _GET + f'\nget "{repo}" "{path}" "{sub}"'], check=False)

# 出图底模（按 §1）
if IMG_KIND and IMG_FILE:
    sub = 'checkpoints' if IMG_KIND == 'checkpoint' else 'unet'
    if IMG_CIVITAI_VERSION and os.environ.get('CIVITAI_TOKEN'):
        q = '?token=' + os.environ['CIVITAI_TOKEN'] + (('&fileId='+IMG_CIVITAI_FILEID) if IMG_CIVITAI_FILEID else '')
        subprocess.run(['bash','-lc',
            f'cd /content/ComfyUI/models/{sub} && [ -s "{IMG_FILE}" ] || wget -q -O "{IMG_FILE}" '
            f'"https://civitai.com/api/download/models/{IMG_CIVITAI_VERSION}{q}"'], check=False)
        print('出图底模(CivitAI) →', IMG_FILE)
    elif IMG_HF_REPO:
        hf_get(IMG_HF_REPO, IMG_FILE, sub); print('出图底模(HF) →', IMG_FILE)
else:
    print('（未配出图底模，跳过）')

# 你清单里的权重
for repo, path, sub in MODELS_TO_DOWNLOAD:
    hf_get(repo, path, sub)
print('下载结束（空清单=未下任何模型；在 §1 加了再 Run all 即补下，已下的会 [skip]）')


In [ ]:
# ===== §5b SageAttention 安装（仅 A100/V100 等非原生 fp8 卡；Blackwell 不装）=====
# §2 给非原生fp8卡设了 MIRAGE_USE_SAGE=1，但要真装上 sageattention，§6 起 ComfyUI 才会加
# --use-sage-attention（否则 find_spec 过不了、等于白标）。A100(sm_80) 有现成预编译轮子，注意力提速明显。
# 装失败不致命：§6 的 find_spec 自动跳过、退回 sdpa。原生 fp8 卡(Blackwell) MIRAGE_USE_SAGE=0 → 这里直接跳过。
import os, sys, subprocess, importlib.util
if os.environ.get('MIRAGE_USE_SAGE', '1') == '1' and importlib.util.find_spec('sageattention') is None:
    print('装 SageAttention（A100 等非原生fp8卡注意力提速）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sageattention'], check=False)
    importlib.invalidate_caches()
    print('SageAttention', '已就绪 → §6 起 ComfyUI 会加 --use-sage-attention'
          if importlib.util.find_spec('sageattention') else '未装上（退回 sdpa，不影响出片）')
else:
    print('SageAttention 跳过（原生fp8卡=关 sage，或已装）。')


In [ ]:
# ===== §6 起 ComfyUI（硬重启；脱离内核，停 cell 不杀）=====
import os, sys, importlib.util
sys.path.insert(0, '/content/mirage/colab'); import persist
_args = ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188']
if os.environ.get('MIRAGE_HIGHVRAM') == '1':
    _args.append('--highvram')
if os.environ.get('MIRAGE_USE_SAGE', '1') == '1' and importlib.util.find_spec('sageattention'):
    _args.append('--use-sage-attention')
print('ComfyUI 启动参数:', ' '.join(_args[2:]))
persist.restart_service('ComfyUI', _args, 'http://127.0.0.1:8188/', '/content/comfyui.log', 8188, 'ComfyUI/main.py', timeout=180)


In [ ]:
# ===== §7 装后端依赖 + 写 .env（按 §1 的出图/出片；没填的就不设）=====
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
lines = [
    'COMFYUI_BASE_URL=http://127.0.0.1:8188',
    'COMFYUI_VIDEO_AS=auto',          # 配了端点就让出片透明走 ComfyUI
    'VIDEO_PROVIDER_DEFAULT=wan2.2',  # 面板出片模型名（实际跑哪套 = 你选的 workflow + 下的权重）
]
if DEEPSEEK_KEY: lines.append('OPENAI_API_KEY=' + DEEPSEEK_KEY)
if VID_WORKFLOW: lines.append('COMFYUI_WORKFLOW_I2V=' + VID_WORKFLOW)
if IMG_KIND == 'checkpoint' and IMG_FILE:
    lines += ['COMFYUI_FLUX_CKPT=' + IMG_FILE,
              'COMFYUI_WORKFLOW_T2I=' + (VID_T2I_WORKFLOW or 'comfyui_workflows/t2i_checkpoint_template.json'),
              'COMFYUI_IMAGE_AS=auto']
elif IMG_KIND == 'unet' and IMG_FILE:
    lines += ['COMFYUI_FLUX_UNET=' + IMG_FILE,
              'COMFYUI_WORKFLOW_T2I=' + VID_T2I_WORKFLOW,   # 空=用仓库自带 flux t2i 模板
              'COMFYUI_IMAGE_AS=auto']
open('.env', 'w').write('\n'.join(l for l in lines if l) + '\n')
print('.env 写好（已设：' + ', '.join(l.split('=')[0] for l in lines if l and not l.startswith('OPENAI')) + '）')
print('  出片 workflow:', VID_WORKFLOW or '(未设，用后端自带默认路径——记得先在 §1 下对应权重)')
print('  出图底模:', (IMG_FILE or '(未配)'))


In [ ]:
# ===== §8 起后端（硬重启，读到 §7 刚写的 .env）=====
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端',
    ['uvicorn','mirage.main_api:app','--host','0.0.0.0','--port','8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn',
    cwd='/content/mirage', timeout=120)


In [ ]:
# ===== §9 cloudflared 暴露后端（脱离内核；秒级打印公网地址）=====
import sys, re, time, os, subprocess
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m: url = m[-1]; break
print('✅ 公网地址:', url) if url else print('⚠ 隧道 60s 未就绪 → !tail -20 /content/cf.log（多半限流，重跑本格）')
print('打开它 → 选工作目录 → 一键出图→选图→出片合成（出片用你 §1 选的模型）')


In [ ]:
# ===== §10 实时日志（看 ComfyUI 在画什么；停本格只停 tail，不杀服务）=====
!tail -n 40 -f /content/comfyui.log


---
## 怎么填 §1（示例，自己挑一套填进去）

**出片 = Wan2.2-A14B（fp8 极速/精修，原生 FP8 卡）**
```python
VID_WORKFLOW = 'comfyui_workflows/i2v_fp8_template.json'
MODELS_TO_DOWNLOAD = [
  ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors','unet'),
  ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors','unet'),
  ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','clip'),
  ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/vae/wan_2.1_vae.safetensors','vae'),
]
```
> A100-80G 不量化 → 把上面 fp8 换成 `..._14B_fp16.safetensors` + `umt5_xxl_fp16.safetensors`，`VID_WORKFLOW='comfyui_workflows/i2v_bf16_template.json'`。
> 极速档：再加 Lightning LoRA 两条到 `loras/`，`VID_WORKFLOW` 指 `i2v_fp8_lightning_template.json`。

**出图 = CivitAI 全合一 checkpoint（无审查等）**
```python
IMG_KIND='checkpoint'; IMG_FILE='my-checkpoint.safetensors'
IMG_CIVITAI_VERSION='你的版本号'   # 右侧🔑 加 CIVITAI_TOKEN
```
**出图 = HF UNET-only（如 FLUX.1-dev，需 HF_TOKEN）**
```python
IMG_KIND='unet'; IMG_FILE='flux1-dev.safetensors'; IMG_HF_REPO='black-forest-labs/FLUX.1-dev'
```

**对口型 / RealESRGAN 放大 / 人物 LoRA 训练** 等是独立环节：模型加到 `MODELS_TO_DOWNLOAD`，对应开关在 `.env`/面板里开；LoRA 训练见仓库自带 `colab_deploy.ipynb` 的 L1–L4。
